# V6 Feature Correlations

This notebook checks feature-level correlations for the same stats used by V6 scoring.

**Quick path:** run the load + prep cells, then **Individual stat effectiveness** — one table with each single-stat rule’s win rate. Use the rest for deeper correlation views.

It mirrors the V6 bullpen fallback logic and then runs one cell per feature.

## Targets (prep cell)

- **`home_win`** — Did the **home** team win? (1/0). A positive correlation with `sp_kbb_diff` means: when home has better K-BB than away, home wins more often. Same idea for other home-minus-away diffs.
- **`favwon_*`** — Did the team favored by **that one stat alone** win? Built with `make_favorite_won_from_feature` (xFIP: lower is better; bullpen: `pen_gap_fip` with missing → 0). Includes **`favwon_pen`** for bullpen-only picks.
- **`risk_adjusted_edge`** — Only if that column exists in the parquet (usually empty here).

## How to read correlations quickly

- **Pearson** ≈ linear association; **Spearman** ≈ monotonic (rank) association. For binary outcomes, both are common; focus on **magnitude** and **sign**, not p-values in this notebook.
- **`home_win` rows** — “Does this feature edge predict the home team winning?” Strong offense/SP edges often show modest positive |r| here; bullpen is often weaker.
- **`favwon_*` vs the *same* underlying stat** — **Partly tautological** (e.g. `sp_kbb_diff` vs `favwon_sp_kbb`): the target was defined from that column. Summary tables include a **“no diagonal”** slice that drops those pairs so cross-target comparisons are easier.
- **`favwon_*` vs a *different* feature** — e.g. `offense_platoon_diff` vs `favwon_sp_kbb`: “Does platoon edge align with whether the K-BB favorite won?” Often near zero; that is normal when single-stat favorites disagree.

## Re-running for different ranges

- Change **`SEASON`** (or point `PARQUET` at another file) in the first code cell.
- Optionally filter **`work`** by `game_date` after the completed-games filter for partial seasons or rolling windows.
- Re-run **top to bottom** so `favwon_*` and summaries stay consistent.

---

The **Summary** cells use `default_corr_targets(work)` so targets stay in sync.


In [84]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)

# Change SEASON or set PARQUET manually to point at another file when comparing years.
SEASON = 2026  # datetime.now().year
FINAL_STATES = {"Final", "Game Over", "Completed Early"}

cwd = Path.cwd()
DATA = cwd / "data" if (cwd / "data").is_dir() else cwd.parent / "data"
PARQUET = DATA / f"games_{SEASON}.parquet"

if not PARQUET.is_file():
    raise FileNotFoundError(f"Missing {PARQUET}. Set SEASON manually or run pipeline.")

df = pd.read_parquet(PARQUET).copy()
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce").dt.normalize()
df["detailed_state"] = df["detailed_state"].astype(str)

# Correlation eval is done on completed games with known home_win.
work = df[df["detailed_state"].isin(FINAL_STATES)].copy()
work = work[work["home_win"].notna()].copy()
# Optional: restrict date range, e.g. second half only or rolling window:
# work = work[work["game_date"] >= pd.Timestamp("2024-06-01")].copy()

print("Using:", PARQUET.resolve())
print("Completed rows with home_win:", len(work))


Using: /Users/austinlolli/Code/baseball/mlb-model/data/games_2026.parquet
Completed rows with home_win: 617


In [85]:
# Build V6 helper columns used for feature correlation comparisons.
# Order: pen gaps → favwon_* columns → default_corr_targets / corr helpers.
# After changing SEASON or filtering `work`, re-run this cell before summaries.

PEN_ABS = 0.75

def preferred_pen_cols(frame: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    if "home_pen_roll14_fip" in frame.columns:
        home_pen = frame["home_pen_roll14_fip"]
        if "home_pen_season_fip" in frame.columns:
            home_pen = home_pen.combine_first(frame["home_pen_season_fip"])
    elif "home_pen_season_fip" in frame.columns:
        home_pen = frame["home_pen_season_fip"]
    else:
        home_pen = pd.Series(np.nan, index=frame.index)

    if "away_pen_roll14_fip" in frame.columns:
        away_pen = frame["away_pen_roll14_fip"]
        if "away_pen_season_fip" in frame.columns:
            away_pen = away_pen.combine_first(frame["away_pen_season_fip"])
    elif "away_pen_season_fip" in frame.columns:
        away_pen = frame["away_pen_season_fip"]
    else:
        away_pen = pd.Series(np.nan, index=frame.index)

    return home_pen, away_pen

home_pen, away_pen = preferred_pen_cols(work)
work["pen_gap_fip"] = away_pen - home_pen
work["pen_norm_v6"] = (work["pen_gap_fip"] / PEN_ABS).fillna(0.0)

# proxy for v6 favorite check, use this to check individual stat split correlations
def make_favorite_won_from_feature(
    frame: pd.DataFrame,
    feature_col: str,
    *,
    higher_is_better_for_home: bool = True,
    target_col: str = "home_win",
) -> pd.Series:
    """
    Build favorite_won for a single feature:
    - prefer_home = feature >= 0 (or <= 0 if higher_is_better_for_home=False)
    - favorite_won = 1 if preferred side actually won, else 0
    """
    feat = pd.to_numeric(frame[feature_col], errors="coerce")
    y = pd.to_numeric(frame[target_col], errors="coerce")

    if higher_is_better_for_home:
        prefer_home = feat >= 0
    else:
        prefer_home = feat <= 0

    fav_won = np.where(prefer_home, y == 1.0, y == 0.0)
    out = pd.Series(fav_won, index=frame.index, dtype="float")
    out[feat.isna() | y.isna()] = np.nan
    return out

work["favwon_sp_kbb"] = make_favorite_won_from_feature(work, "sp_kbb_diff")
work["favwon_sp_xfip"] = make_favorite_won_from_feature(work, "sp_xfip_diff", higher_is_better_for_home=False)
work["favwon_off"] = make_favorite_won_from_feature(work, "offense_diff")
work["favwon_platoon"] = make_favorite_won_from_feature(work, "offense_platoon_diff")

# Bullpen-only favorite: same direction as V6 — pen_gap_fip = away_FIP − home_FIP (lower FIP better).
# Prefer home when gap > 0 (home pen better). Missing gap → 0 (neutral pen), same as V6 fill.
_tmp_pen = work.copy()
_tmp_pen["_pen_gap_rule"] = pd.to_numeric(_tmp_pen["pen_gap_fip"], errors="coerce").fillna(0.0)
work["favwon_pen"] = make_favorite_won_from_feature(_tmp_pen, "_pen_gap_rule", higher_is_better_for_home=True)


def default_corr_targets(frame: pd.DataFrame) -> list[str]:
    """Targets for corr_table / show_feature_corr when targets=None."""
    out = [
        "home_win",
        "favwon_sp_kbb",
        "favwon_sp_xfip",
        "favwon_off",
        "favwon_platoon",
        "favwon_pen",
    ]
    if "risk_adjusted_edge" in frame.columns:
        out.append("risk_adjusted_edge")
    return out


# Which favwon_* column is built from which raw stat (for dropping tautological correlation rows).
FEATURE_TO_FAVWON_COL: dict[str, str] = {
    "sp_kbb_diff": "favwon_sp_kbb",
    "sp_xfip_diff": "favwon_sp_xfip",
    "offense_diff": "favwon_off",
    "offense_platoon_diff": "favwon_platoon",
    "pen_gap_fip": "favwon_pen",
    "pen_norm_v6": "favwon_pen",
}


def is_tautological_corr(feature: str, target: str) -> bool:
    """True when correlating a stat with the favwon_* target derived from that same stat."""
    return FEATURE_TO_FAVWON_COL.get(feature) == target


def drop_tautological_corr_rows(corr_df: pd.DataFrame) -> pd.DataFrame:
    """Remove diagonal favwon_* pairs so summaries emphasize cross-stat comparisons."""
    if corr_df.empty or "feature" not in corr_df.columns or "target" not in corr_df.columns:
        return corr_df
    keep = ~corr_df.apply(
        lambda r: is_tautological_corr(str(r["feature"]), str(r["target"])),
        axis=1,
    )
    return corr_df.loc[keep].reset_index(drop=True)


def corr_table(frame: pd.DataFrame, feature: str, targets: list[str] | None = None) -> pd.DataFrame:
    if targets is None:
        targets = default_corr_targets(frame)

    rows = []
    for tgt in targets:
        if tgt not in frame.columns:
            rows.append({"feature": feature, "target": tgt, "n": 0, "pearson": np.nan, "spearman": np.nan})
            continue

        d = frame[[feature, tgt]].copy()
        d[feature] = pd.to_numeric(d[feature], errors="coerce")
        d[tgt] = pd.to_numeric(d[tgt], errors="coerce")
        d = d.dropna()

        if len(d) < 3:
            rows.append({"feature": feature, "target": tgt, "n": int(len(d)), "pearson": np.nan, "spearman": np.nan})
            continue

        rows.append(
            {
                "feature": feature,
                "target": tgt,
                "n": int(len(d)),
                "pearson": float(d[feature].corr(d[tgt], method="pearson")),
                "spearman": float(d[feature].corr(d[tgt], method="spearman")),
            }
        )
    return pd.DataFrame(rows)

def show_feature_corr(
    feature: str,
    *,
    frame: pd.DataFrame | None = None,
    targets: list[str] | None = None,
    include_describe: bool = True,
) -> pd.DataFrame:
    if frame is None:
        frame = work
    if targets is None:
        targets = default_corr_targets(frame)

    print(f"\n=== {feature} correlations ===")
    out = corr_table(frame, feature, targets=targets)
    display(out)

    if include_describe:
        cols = [c for c in [feature] + targets if c in frame.columns]
        if cols:
            display(frame[cols].dropna().describe().T)

    return out


## Individual stat effectiveness (start here)

Each row is a **single-stat rule**: pick the side implied by the home-minus-away diff (xFIP uses “lower is better”; bullpen uses **away FIP − home FIP**, prefer home when that gap **>** 0). **Favorite win rate** = how often that pick won.

- **`n`** = games with a defined `favwon_*`. For SP/offense, missing raw stats reduce **n**. **Bullpen** matches V6: missing `pen_gap_fip` is treated as **0** (neutral pen), so **n** is usually the full completed sample.
- **`se`** = rough standard error for a proportion, \(\sqrt{p(1-p)/n}\).
- Compare to **baseline**: always picking the home team = `mean(home_win)` in this sample; random = 0.50.

The correlation sections below add *how* edges co-move with outcomes; this table answers *how often each rule is right* in one glance.

In [86]:
# --- Single-stat favorite win rates (effectiveness at a glance) ---
rules = [
    ("sp_kbb_diff", "favwon_sp_kbb", "SP K-BB"),
    ("sp_xfip_diff", "favwon_sp_xfip", "SP xFIP"),
    ("offense_diff", "favwon_off", "Offense (wRC+)"),
    ("offense_platoon_diff", "favwon_platoon", "Offense platoon"),
    ("pen_gap_fip", "favwon_pen", "Bullpen (FIP gap)"),
]

rows = []
for _raw, fav_col, label in rules:
    if fav_col not in work.columns:
        continue
    s = pd.to_numeric(work[fav_col], errors="coerce").dropna()
    n = int(len(s))
    if n == 0:
        continue
    p = float(s.mean())
    se = float(np.sqrt(p * (1.0 - p) / n))
    rows.append(
        {
            "rule": label,
            "favwon_col": fav_col,
            "n": n,
            "favorite_win_rate": p,
            "se": se,
        }
    )

effectiveness = pd.DataFrame(rows).sort_values("favorite_win_rate", ascending=False).reset_index(drop=True)
display(effectiveness)

hw = float(pd.to_numeric(work["home_win"], errors="coerce").mean())
print(f"Baseline — always pick home: mean(home_win) = {hw:.4f}  |  coin flip = 0.5000")

,rule,favwon_col,n,favorite_win_rate,se
0,Offense (wRC+),favwon_off,617,0.542950,0.020055
1,Offense platoon,favwon_platoon,608,0.542763,0.020203
2,SP K-BB,favwon_sp_kbb,548,0.536496,0.021302
3,Bullpen (FIP gap),favwon_pen,617,0.520259,0.020113
4,SP xFIP,favwon_sp_xfip,547,0.508227,0.021376


Baseline — always pick home: mean(home_win) = 0.5802  |  coin flip = 0.5000


In [66]:
# 1) SP K-BB differential (V6: sp_kbb_diff)
show_feature_corr("sp_kbb_diff")



=== sp_kbb_diff correlations ===


,feature,target,n,pearson,spearman
0,sp_kbb_diff,home_win,2933,0.136635,0.134587
1,sp_kbb_diff,favwon_sp_kbb,2933,0.010029,0.012731
2,sp_kbb_diff,favwon_sp_xfip,2933,0.010333,0.012512
3,sp_kbb_diff,favwon_off,2933,0.012203,0.016665
4,sp_kbb_diff,favwon_platoon,2933,0.040649,0.039194


,count,mean,std,min,25%,50%,75%,max
sp_kbb_diff,2933.0,0.161661,7.844466,-34.695341,-4.760952,0.015133,4.895105,50.36075
home_win,2933.0,0.511763,0.499947,0.000000,0.000000,1.000000,1.000000,1.00000
favwon_sp_kbb,2933.0,0.560518,0.496409,0.000000,0.000000,1.000000,1.000000,1.00000
favwon_sp_xfip,2933.0,0.553358,0.497230,0.000000,0.000000,1.000000,1.000000,1.00000
favwon_off,2933.0,0.555063,0.497044,0.000000,0.000000,1.000000,1.000000,1.00000
favwon_platoon,2933.0,0.558814,0.496614,0.000000,0.000000,1.000000,1.000000,1.00000


,feature,target,n,pearson,spearman
0,sp_kbb_diff,home_win,2933,0.136635,0.134587
1,sp_kbb_diff,favwon_sp_kbb,2933,0.010029,0.012731
2,sp_kbb_diff,favwon_sp_xfip,2933,0.010333,0.012512
3,sp_kbb_diff,favwon_off,2933,0.012203,0.016665
4,sp_kbb_diff,favwon_platoon,2933,0.040649,0.039194


In [67]:
# 2) SP xFIP differential (V6: sp_xfip_diff)
show_feature_corr("sp_xfip_diff")



=== sp_xfip_diff correlations ===


,feature,target,n,pearson,spearman
0,sp_xfip_diff,home_win,2933,-0.106577,-0.138783
1,sp_xfip_diff,favwon_sp_kbb,2933,-0.017624,-0.019197
2,sp_xfip_diff,favwon_sp_xfip,2933,-0.015981,-0.020919
3,sp_xfip_diff,favwon_off,2933,-0.001460,-0.010980
4,sp_xfip_diff,favwon_platoon,2933,-0.019040,-0.032710


,count,mean,std,min,25%,50%,75%,max
sp_xfip_diff,2933.0,-0.052681,1.631240,-35.125,-0.773069,-0.002863,0.75127,7.972638
home_win,2933.0,0.511763,0.499947,0.000,0.000000,1.000000,1.00000,1.000000
favwon_sp_kbb,2933.0,0.560518,0.496409,0.000,0.000000,1.000000,1.00000,1.000000
favwon_sp_xfip,2933.0,0.553358,0.497230,0.000,0.000000,1.000000,1.00000,1.000000
favwon_off,2933.0,0.555063,0.497044,0.000,0.000000,1.000000,1.00000,1.000000
favwon_platoon,2933.0,0.558814,0.496614,0.000,0.000000,1.000000,1.00000,1.000000


,feature,target,n,pearson,spearman
0,sp_xfip_diff,home_win,2933,-0.106577,-0.138783
1,sp_xfip_diff,favwon_sp_kbb,2933,-0.017624,-0.019197
2,sp_xfip_diff,favwon_sp_xfip,2933,-0.015981,-0.020919
3,sp_xfip_diff,favwon_off,2933,-0.001460,-0.010980
4,sp_xfip_diff,favwon_platoon,2933,-0.019040,-0.032710


In [68]:
# 3) Offense differential (V6: offense_diff)
show_feature_corr("offense_diff")



=== offense_diff correlations ===


,feature,target,n,pearson,spearman
0,offense_diff,home_win,2959,0.140919,0.136286
1,offense_diff,favwon_sp_kbb,2933,0.025748,0.022917
2,offense_diff,favwon_sp_xfip,2933,0.031408,0.026970
3,offense_diff,favwon_off,2959,0.013748,0.013933
4,offense_diff,favwon_platoon,2954,0.004321,0.002462


,count,mean,std,min,25%,50%,75%,max
offense_diff,2933.0,0.014054,7.041430,-22.930832,-5.064478,0.0,4.783118,22.368113
home_win,2933.0,0.511763,0.499947,0.000000,0.000000,1.0,1.000000,1.000000
favwon_sp_kbb,2933.0,0.560518,0.496409,0.000000,0.000000,1.0,1.000000,1.000000
favwon_sp_xfip,2933.0,0.553358,0.497230,0.000000,0.000000,1.0,1.000000,1.000000
favwon_off,2933.0,0.555063,0.497044,0.000000,0.000000,1.0,1.000000,1.000000
favwon_platoon,2933.0,0.558814,0.496614,0.000000,0.000000,1.0,1.000000,1.000000


,feature,target,n,pearson,spearman
0,offense_diff,home_win,2959,0.140919,0.136286
1,offense_diff,favwon_sp_kbb,2933,0.025748,0.022917
2,offense_diff,favwon_sp_xfip,2933,0.031408,0.026970
3,offense_diff,favwon_off,2959,0.013748,0.013933
4,offense_diff,favwon_platoon,2954,0.004321,0.002462


In [69]:
# 4) Offense platoon differential (V6: offense_platoon_diff)
show_feature_corr("offense_platoon_diff")



=== offense_platoon_diff correlations ===


,feature,target,n,pearson,spearman
0,offense_platoon_diff,home_win,2954,0.152404,0.146704
1,offense_platoon_diff,favwon_sp_kbb,2933,0.034597,0.036285
2,offense_platoon_diff,favwon_sp_xfip,2933,0.048457,0.044629
3,offense_platoon_diff,favwon_off,2954,0.005172,0.006139
4,offense_platoon_diff,favwon_platoon,2954,0.010398,0.012369


,count,mean,std,min,25%,50%,75%,max
offense_platoon_diff,2933.0,-0.036382,7.926084,-26.064904,-5.620346,0.140509,5.19882,23.231477
home_win,2933.0,0.511763,0.499947,0.000000,0.000000,1.000000,1.00000,1.000000
favwon_sp_kbb,2933.0,0.560518,0.496409,0.000000,0.000000,1.000000,1.00000,1.000000
favwon_sp_xfip,2933.0,0.553358,0.497230,0.000000,0.000000,1.000000,1.00000,1.000000
favwon_off,2933.0,0.555063,0.497044,0.000000,0.000000,1.000000,1.00000,1.000000
favwon_platoon,2933.0,0.558814,0.496614,0.000000,0.000000,1.000000,1.00000,1.000000


,feature,target,n,pearson,spearman
0,offense_platoon_diff,home_win,2954,0.152404,0.146704
1,offense_platoon_diff,favwon_sp_kbb,2933,0.034597,0.036285
2,offense_platoon_diff,favwon_sp_xfip,2933,0.048457,0.044629
3,offense_platoon_diff,favwon_off,2954,0.005172,0.006139
4,offense_platoon_diff,favwon_platoon,2954,0.010398,0.012369


In [70]:
# 5) Bullpen gap from V6-preferred FIP source (away - home)
show_feature_corr("pen_gap_fip")



=== pen_gap_fip correlations ===


,feature,target,n,pearson,spearman
0,pen_gap_fip,home_win,2581,0.044796,0.052911
1,pen_gap_fip,favwon_sp_kbb,2579,0.004939,0.002637
2,pen_gap_fip,favwon_sp_xfip,2579,-0.010018,-0.013986
3,pen_gap_fip,favwon_off,2581,0.009593,0.011723
4,pen_gap_fip,favwon_platoon,2579,-0.005540,-0.000637


,count,mean,std,min,25%,50%,75%,max
pen_gap_fip,2579.0,0.026906,1.571900,-5.815524,-1.008053,0.022063,1.009642,15.785714
home_win,2579.0,0.513765,0.499907,0.000000,0.000000,1.000000,1.000000,1.000000
favwon_sp_kbb,2579.0,0.566499,0.495654,0.000000,0.000000,1.000000,1.000000,1.000000
favwon_sp_xfip,2579.0,0.558744,0.496633,0.000000,0.000000,1.000000,1.000000,1.000000
favwon_off,2579.0,0.559519,0.496541,0.000000,0.000000,1.000000,1.000000,1.000000
favwon_platoon,2579.0,0.562621,0.496159,0.000000,0.000000,1.000000,1.000000,1.000000


,feature,target,n,pearson,spearman
0,pen_gap_fip,home_win,2581,0.044796,0.052911
1,pen_gap_fip,favwon_sp_kbb,2579,0.004939,0.002637
2,pen_gap_fip,favwon_sp_xfip,2579,-0.010018,-0.013986
3,pen_gap_fip,favwon_off,2581,0.009593,0.011723
4,pen_gap_fip,favwon_platoon,2579,-0.005540,-0.000637


In [71]:
# 6) Bullpen normalized signal used by V6 (pen_gap_fip / PEN_ABS)
show_feature_corr("pen_norm_v6")



=== pen_norm_v6 correlations ===


,feature,target,n,pearson,spearman
0,pen_norm_v6,home_win,2959,0.041897,0.050938
1,pen_norm_v6,favwon_sp_kbb,2933,0.004818,0.002966
2,pen_norm_v6,favwon_sp_xfip,2933,-0.009208,-0.012023
3,pen_norm_v6,favwon_off,2959,0.009090,0.008510
4,pen_norm_v6,favwon_platoon,2954,-0.005040,-0.001383


,count,mean,std,min,25%,50%,75%,max
pen_norm_v6,2933.0,0.031545,1.965308,-7.754032,-1.115207,0.0,1.138811,21.047619
home_win,2933.0,0.511763,0.499947,0.000000,0.000000,1.0,1.000000,1.000000
favwon_sp_kbb,2933.0,0.560518,0.496409,0.000000,0.000000,1.0,1.000000,1.000000
favwon_sp_xfip,2933.0,0.553358,0.497230,0.000000,0.000000,1.0,1.000000,1.000000
favwon_off,2933.0,0.555063,0.497044,0.000000,0.000000,1.0,1.000000,1.000000
favwon_platoon,2933.0,0.558814,0.496614,0.000000,0.000000,1.0,1.000000,1.000000


,feature,target,n,pearson,spearman
0,pen_norm_v6,home_win,2959,0.041897,0.050938
1,pen_norm_v6,favwon_sp_kbb,2933,0.004818,0.002966
2,pen_norm_v6,favwon_sp_xfip,2933,-0.009208,-0.012023
3,pen_norm_v6,favwon_off,2959,0.009090,0.008510
4,pen_norm_v6,favwon_platoon,2954,-0.005040,-0.001383


In [72]:
# Summary: all V6 features × all default targets
# Full table includes tautological pairs (e.g. sp_kbb_diff × favwon_sp_kbb). The "no diagonal" table
# drops those so you can rank cross-stat / home_win relationships without inflated same-stat rows.

SUMMARY_TARGETS = default_corr_targets(work)

features = [
    "sp_kbb_diff",
    "sp_xfip_diff",
    "offense_diff",
    "offense_platoon_diff",
    "pen_gap_fip",
    "pen_norm_v6",
]

all_corr = []
for feat in features:
    if feat in work.columns:
        all_corr.append(corr_table(work, feat, targets=SUMMARY_TARGETS))

if all_corr:
    all_corr_df = (
        pd.concat(all_corr, ignore_index=True)
        .assign(abs_pearson=lambda d: d["pearson"].abs(), abs_spearman=lambda d: d["spearman"].abs())
    )
    print("Full ranked table (includes same-stat favwon_* diagonal)")
    display(
        all_corr_df.sort_values(["target", "abs_pearson"], ascending=[True, False])[
            ["feature", "target", "n", "pearson", "spearman", "abs_pearson", "abs_spearman"]
        ]
    )
    no_diag = drop_tautological_corr_rows(all_corr_df)
    print("Ranked table — same-stat favwon_* pairs removed (cross-target view)")
    display(
        no_diag.sort_values(["target", "abs_pearson"], ascending=[True, False])[
            ["feature", "target", "n", "pearson", "spearman", "abs_pearson", "abs_spearman"]
        ]
    )
else:
    print("No features found in work for summary table.")


Full ranked table (includes same-stat favwon_* diagonal)


,feature,target,n,pearson,spearman,abs_pearson,abs_spearman
13,offense_diff,favwon_off,2959,0.013748,0.013933,0.013748,0.013933
3,sp_kbb_diff,favwon_off,2933,0.012203,0.016665,0.012203,0.016665
23,pen_gap_fip,favwon_off,2581,0.009593,0.011723,0.009593,0.011723
28,pen_norm_v6,favwon_off,2959,0.009090,0.008510,0.009090,0.008510
18,offense_platoon_diff,favwon_off,2954,0.005172,0.006139,0.005172,0.006139
8,sp_xfip_diff,favwon_off,2933,-0.001460,-0.010980,0.001460,0.010980
4,sp_kbb_diff,favwon_platoon,2933,0.040649,0.039194,0.040649,0.039194
9,sp_xfip_diff,favwon_platoon,2933,-0.019040,-0.032710,0.019040,0.032710
19,offense_platoon_diff,favwon_platoon,2954,0.010398,0.012369,0.010398,0.012369
24,pen_gap_fip,favwon_platoon,2579,-0.005540,-0.000637,0.005540,0.000637


Ranked table — same-stat favwon_* pairs removed (cross-target view)


,feature,target,n,pearson,spearman,abs_pearson,abs_spearman
2,sp_kbb_diff,favwon_off,2933,0.012203,0.016665,0.012203,0.016665
19,pen_gap_fip,favwon_off,2581,0.009593,0.011723,0.009593,0.011723
24,pen_norm_v6,favwon_off,2959,0.009090,0.008510,0.009090,0.008510
15,offense_platoon_diff,favwon_off,2954,0.005172,0.006139,0.005172,0.006139
6,sp_xfip_diff,favwon_off,2933,-0.001460,-0.010980,0.001460,0.010980
3,sp_kbb_diff,favwon_platoon,2933,0.040649,0.039194,0.040649,0.039194
7,sp_xfip_diff,favwon_platoon,2933,-0.019040,-0.032710,0.019040,0.032710
20,pen_gap_fip,favwon_platoon,2579,-0.005540,-0.000637,0.005540,0.000637
25,pen_norm_v6,favwon_platoon,2954,-0.005040,-0.001383,0.005040,0.001383
11,offense_diff,favwon_platoon,2954,0.004321,0.002462,0.004321,0.002462


In [73]:
# Optional: compare one feature against custom favorite targets in a single run

custom_targets = default_corr_targets(work)

feature_to_check = "offense_platoon_diff"
_ = show_feature_corr(
    feature_to_check,
    targets=custom_targets,
    include_describe=False,
)



=== offense_platoon_diff correlations ===


,feature,target,n,pearson,spearman
0,offense_platoon_diff,home_win,2954,0.152404,0.146704
1,offense_platoon_diff,favwon_sp_kbb,2933,0.034597,0.036285
2,offense_platoon_diff,favwon_sp_xfip,2933,0.048457,0.044629
3,offense_platoon_diff,favwon_off,2954,0.005172,0.006139
4,offense_platoon_diff,favwon_platoon,2954,0.010398,0.012369


In [74]:
# Pivot + ranked views (same data as Summary cell; pivots are easier to scan)
# Use custom_corr_no_diag pivots when comparing favwon_* columns — diagonal cells are tautological.

features = [
    "sp_kbb_diff",
    "sp_xfip_diff",
    "offense_diff",
    "offense_platoon_diff",
    "pen_gap_fip",
    "pen_norm_v6",
]

custom_targets = default_corr_targets(work)

rows = []
for feat in features:
    if feat not in work.columns:
        continue
    rows.append(corr_table(work, feat, targets=custom_targets))

if not rows:
    print("No feature/target rows available for custom summary.")
else:
    custom_corr = pd.concat(rows, ignore_index=True)
    custom_corr["abs_pearson"] = custom_corr["pearson"].abs()
    custom_corr["abs_spearman"] = custom_corr["spearman"].abs()
    custom_corr_no_diag = drop_tautological_corr_rows(custom_corr)

    print("Ranked detail table (full)")
    display(
        custom_corr.sort_values(["target", "abs_pearson"], ascending=[True, False])[
            ["feature", "target", "n", "pearson", "spearman", "abs_pearson", "abs_spearman"]
        ]
    )
    print("Ranked detail — same-stat favwon_* pairs removed")
    display(
        custom_corr_no_diag.sort_values(["target", "abs_pearson"], ascending=[True, False])[
            ["feature", "target", "n", "pearson", "spearman", "abs_pearson", "abs_spearman"]
        ]
    )

    print("\nPivot: Pearson (full; diagonal favwon cells may be inflated)")
    pearson_pivot = custom_corr.pivot(index="feature", columns="target", values="pearson")
    display(pearson_pivot)

    print("\nPivot: Pearson — tautological favwon cells set to NaN for heatmap-style reading")
    pearson_pivot_clean = pearson_pivot.copy()
    for f, fav_col in FEATURE_TO_FAVWON_COL.items():
        if f in pearson_pivot_clean.index and fav_col in pearson_pivot_clean.columns:
            pearson_pivot_clean.loc[f, fav_col] = np.nan
    display(pearson_pivot_clean)

    print("\nPivot: |Pearson| (full), sorted by home_win strength")
    abs_pearson_pivot = custom_corr.pivot(index="feature", columns="target", values="abs_pearson")
    display(abs_pearson_pivot.sort_values("home_win", ascending=False))

    print("\nPivot: |Pearson| — diagonal NaN (same as pearson_pivot_clean)")
    abs_clean = abs_pearson_pivot.copy()
    for f, fav_col in FEATURE_TO_FAVWON_COL.items():
        if f in abs_clean.index and fav_col in abs_clean.columns:
            abs_clean.loc[f, fav_col] = np.nan
    display(abs_clean.sort_values("home_win", ascending=False))

Ranked detail table (full)


,feature,target,n,pearson,spearman,abs_pearson,abs_spearman
13,offense_diff,favwon_off,2959,0.013748,0.013933,0.013748,0.013933
3,sp_kbb_diff,favwon_off,2933,0.012203,0.016665,0.012203,0.016665
23,pen_gap_fip,favwon_off,2581,0.009593,0.011723,0.009593,0.011723
28,pen_norm_v6,favwon_off,2959,0.009090,0.008510,0.009090,0.008510
18,offense_platoon_diff,favwon_off,2954,0.005172,0.006139,0.005172,0.006139
8,sp_xfip_diff,favwon_off,2933,-0.001460,-0.010980,0.001460,0.010980
4,sp_kbb_diff,favwon_platoon,2933,0.040649,0.039194,0.040649,0.039194
9,sp_xfip_diff,favwon_platoon,2933,-0.019040,-0.032710,0.019040,0.032710
19,offense_platoon_diff,favwon_platoon,2954,0.010398,0.012369,0.010398,0.012369
24,pen_gap_fip,favwon_platoon,2579,-0.005540,-0.000637,0.005540,0.000637


Ranked detail — same-stat favwon_* pairs removed


,feature,target,n,pearson,spearman,abs_pearson,abs_spearman
2,sp_kbb_diff,favwon_off,2933,0.012203,0.016665,0.012203,0.016665
19,pen_gap_fip,favwon_off,2581,0.009593,0.011723,0.009593,0.011723
24,pen_norm_v6,favwon_off,2959,0.009090,0.008510,0.009090,0.008510
15,offense_platoon_diff,favwon_off,2954,0.005172,0.006139,0.005172,0.006139
6,sp_xfip_diff,favwon_off,2933,-0.001460,-0.010980,0.001460,0.010980
3,sp_kbb_diff,favwon_platoon,2933,0.040649,0.039194,0.040649,0.039194
7,sp_xfip_diff,favwon_platoon,2933,-0.019040,-0.032710,0.019040,0.032710
20,pen_gap_fip,favwon_platoon,2579,-0.005540,-0.000637,0.005540,0.000637
25,pen_norm_v6,favwon_platoon,2954,-0.005040,-0.001383,0.005040,0.001383
11,offense_diff,favwon_platoon,2954,0.004321,0.002462,0.004321,0.002462



Pivot: Pearson (full; diagonal favwon cells may be inflated)


...

target,favwon_off,favwon_platoon,favwon_sp_kbb,favwon_sp_xfip,home_win
feature,,,,,
offense_diff,NaN,0.004321,0.025748,0.031408,0.140919
offense_platoon_diff,0.005172,NaN,0.034597,0.048457,0.152404
pen_gap_fip,0.009593,-0.005540,0.004939,-0.010018,0.044796
pen_norm_v6,0.009090,-0.005040,0.004818,-0.009208,0.041897
sp_kbb_diff,0.012203,0.040649,NaN,0.010333,0.136635
sp_xfip_diff,-0.001460,-0.019040,-0.017624,NaN,-0.106577



Pivot: |Pearson| (full), sorted by home_win strength


target,favwon_off,favwon_platoon,favwon_sp_kbb,favwon_sp_xfip,home_win
feature,,,,,
offense_platoon_diff,0.005172,0.010398,0.034597,0.048457,0.152404
offense_diff,0.013748,0.004321,0.025748,0.031408,0.140919
sp_kbb_diff,0.012203,0.040649,0.010029,0.010333,0.136635
sp_xfip_diff,0.001460,0.019040,0.017624,0.015981,0.106577
pen_gap_fip,0.009593,0.005540,0.004939,0.010018,0.044796
pen_norm_v6,0.009090,0.005040,0.004818,0.009208,0.041897



Pivot: |Pearson| — diagonal NaN (same as pearson_pivot_clean)


target,favwon_off,favwon_platoon,favwon_sp_kbb,favwon_sp_xfip,home_win
feature,,,,,
offense_platoon_diff,0.005172,NaN,0.034597,0.048457,0.152404
offense_diff,NaN,0.004321,0.025748,0.031408,0.140919
sp_kbb_diff,0.012203,0.040649,NaN,0.010333,0.136635
sp_xfip_diff,0.001460,0.019040,0.017624,NaN,0.106577
pen_gap_fip,0.009593,0.005540,0.004939,0.010018,0.044796
pen_norm_v6,0.009090,0.005040,0.004818,0.009208,0.041897
